# 08 Artefact classifier

For a small share of departures the recorded off-block equals the scheduled time, so the label is simply the delay
(`MVT − SCHED`) instead of a taxi time. This notebook estimates `p_artifact`, the probability that the label equals the
delay. Only departures with a delay above 2000 s are classified; below that limit such a label is usually a genuine
taxi time and the delay itself a plausible one, so `p_artifact` is 0 there.

In [1]:
OOF_MONTHS = None              # None = every month of 2025
ARTEFACT_TOLERANCE_SEC = 60    # label within 60 s of the delay counts as artefact
CANDIDATE_MIN_DELAY_SEC = 2000

## Features

In [2]:
CATEGORICAL = ["ADEP_mvt", "STAND_mvt", "RUNWAY_mvt", "ADES_mvt", "AIRCRAFT_OPERATOR_flt", "AIRCRAFT_TYPE_mvt",
               "MARKET_SEGMENT_flt", "carrier"]
NUMERIC = [
    # schedule and NM times
    "delay_sec", "mvt_minus_aobt_sec", "aobt_missing", "nm_missing", "mvt_minus_eobt_sec", "mvt_minus_iobt_sec",
    "mvt_minus_arvt1_sec", "mvt_minus_arvt3_sec", "sched_minus_eobt_sec", "sched_minus_iobt_sec",
    "aobt_minus_eobt_sec", "aobt_minus_sched_sec", "mvt_hour",
    # ADS-B
    "adsb_movement_to_mvt_sec", "adsb_movement_to_gate_direct_m",
    # stand and inbound aircraft
    "stand_reocc_gap_sec", "stand_inbound_minus_sched_sec", "stand_first_arr_after_sched_gap_sec",
    "stand_n_arr_sched_to_mvt", "stand_prev_arr_gap_sec", "stand_reocc_other_type", "arr_taxi_in_last_hour_sec",
]
FEATURES = CATEGORICAL + NUMERIC

## Model

In [3]:
from catboost import CatBoostClassifier

MODEL_PARAMS = dict(iterations=1500, depth=6, learning_rate=0.05, loss_function="Logloss", random_seed=42,
                    verbose=False, allow_writing_files=False)

## Data

In [4]:
from pathlib import Path
import sys
import tempfile

import joblib
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET, TIME

runner.configure_tracking()
data = pd.read_parquet(ROOT / "data/03_features/train_DEP_merged_sorted.parquet", columns=FEATURES + [TARGET, TIME])
data = data.loc[pd.to_datetime(data[TIME], utc=True).dt.year.eq(2025)].reset_index(drop=True)
month = pd.to_datetime(data[TIME], utc=True).dt.month.to_numpy()
artefact = ((data[TARGET] - data["delay_sec"]).abs() <= ARTEFACT_TOLERANCE_SEC).astype(int).to_numpy()
candidate = (data["delay_sec"] > CANDIDATE_MIN_DELAY_SEC).to_numpy()
print(f"{candidate.sum():,} candidates in 2025, {artefact[candidate].mean():.1%} of them artefacts")

613,095 candidates in 2025, 1.2% of them artefacts


In [5]:
def inputs(frame):
    X = frame[FEATURES].copy()
    X[CATEGORICAL] = X[CATEGORICAL].astype("string").fillna("__MISSING__").astype(object)
    return X


def fit(rows):
    model = CatBoostClassifier(**MODEL_PARAMS)
    model.fit(inputs(data.iloc[rows]), artefact[rows], cat_features=CATEGORICAL)
    return model


def p_artifact(model, frame):
    p = np.zeros(len(frame))
    rows = (frame["delay_sec"] > CANDIDATE_MIN_DELAY_SEC).to_numpy()
    if rows.any():
        p[rows] = model.predict_proba(inputs(frame.loc[rows]))[:, 1]
    return p

## Month blocks

Each month is classified by a model trained on the candidates of the other months.

In [6]:
blocks = runner.month_blocks(OOF_MONTHS)
oof = pd.DataFrame({"row_id": np.arange(len(data)), "month": month, "art": artefact, "candidate": candidate})
oof = oof.loc[np.isin(month, sum(blocks.values(), []))].copy()
oof["fold"] = [f"m{m:02d}" for m in oof["month"]]
oof["p_artifact"] = np.nan
for name, months in blocks.items():
    model = fit(np.flatnonzero(candidate & ~np.isin(month, months)))
    rows = oof.index[oof["fold"].eq(name)]
    oof.loc[rows, "p_artifact"] = p_artifact(model, data.loc[oof.loc[rows, "row_id"]])
    print(name, "done", flush=True)

m01 done


m02 done


m03 done


m04 done


m05 done


...

m08 done


m09 done


m10 done


m11 done


m12 done


In [7]:
cand = oof[oof["candidate"]]
quality = cand.groupby("month").apply(lambda g: pd.Series({
    "candidates": len(g), "artefacts": int(g["art"].sum()),
    "auc": roc_auc_score(g["art"], g["p_artifact"]), "average precision": average_precision_score(g["art"], g["p_artifact"])}),
    include_groups=False)
display(quality.round(3))
print("Calibration over all months:")
cand.groupby(pd.cut(cand["p_artifact"], [0, .02, .1, .3, .5, .7, .9, 1], include_lowest=True), observed=True) \
    .agg(n=("art", "size"), artefact_rate=("art", "mean"), mean_p=("p_artifact", "mean")).round(3)

,candidates,artefacts,auc,average precision
month,,,,
1,40413.0,416.0,0.954,0.364
2,31796.0,544.0,0.939,0.361
3,34111.0,419.0,0.959,0.479
4,39226.0,475.0,0.957,0.545
5,49325.0,550.0,0.968,0.547
6,59101.0,656.0,0.961,0.501
7,77647.0,968.0,0.961,0.510
8,66396.0,784.0,0.965,0.555
9,67075.0,638.0,0.970,0.538


Calibration over all months:


,n,artefact_rate,mean_p
p_artifact,,,
"(-0.001, 0.02]",559749,0.002,0.002
"(0.02, 0.1]",40447,0.051,0.044
"(0.1, 0.3]",9555,0.157,0.155
"(0.3, 0.5]",902,0.341,0.376
"(0.5, 0.7]",353,0.538,0.598
"(0.7, 0.9]",428,0.727,0.808
"(0.9, 1.0]",1661,0.974,0.979


## Final classifier and MLflow run

Trained on all 2025 candidates; `09_corrector` copies it to `model/final`, where the prediction notebooks (`10`, `11`)
read it.

In [8]:
final = fit(np.flatnonzero(candidate))
experiment = mlflow.set_experiment(runner.EXPERIMENT_NAME)
with runner._start_numbered_run(experiment.experiment_id, "Classifier_CatBoost") as run, \
        tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    mlflow.log_params({"model_type": "ArtifactBlocks", "classifier_type": "CatBoostClassifier", "n_blocks": len(blocks),
                       "artifact_tolerance_sec": ARTEFACT_TOLERANCE_SEC, "candidate_min_delay_sec": CANDIDATE_MIN_DELAY_SEC,
                       **{f"clf.{k}": v for k, v in MODEL_PARAMS.items()}})
    mlflow.log_dict(blocks, "block_months.json")
    mlflow.log_dict(FEATURES, "classifier_features.json")
    mlflow.log_dict(CATEGORICAL, "classifier_categorical.json")
    mlflow.log_metrics({f"clf_auc_m{m:02d}": v for m, v in quality["auc"].items()})
    oof.to_parquet(directory / "oof_predictions.parquet", index=False)
    (directory / "models/final").mkdir(parents=True)
    joblib.dump({"model": final, "features": FEATURES, "categorical": CATEGORICAL,
                 "candidate_min_delay_sec": CANDIDATE_MIN_DELAY_SEC}, directory / "models/final/classifier.joblib")
    mlflow.log_artifacts(directory)
print("Run:", run.info.run_name, run.info.run_id)

Run: 294_Classifier_CatBoost 0645226cdf6d4b3eadbe7dc06e028ffa
